# Assignment 2 - Problem 2

## Part a

In [ ]:
import pandas as pd

df = pd.read_csv("retail.csv")
df.head()

,Quantity,StockCode,CustomerID
0,6,85123A,17850.0
1,6,71053,17850.0
2,8,84406B,17850.0
3,6,84029G,17850.0
4,6,84029E,17850.0


In [ ]:
from sklearn.model_selection import train_test_split

# 1. Explore cardinality of the features
unique_stock_codes = df['StockCode'].nunique()
unique_customer_ids = df['CustomerID'].nunique()

print(f"Number of unique StockCodes: {unique_stock_codes}")
print(f"Number of unique CustomerIDs: {unique_customer_ids}")

print("----------")

# 2. Split the data into train, validation, and test sets (60% Train, 20% Val, 20% Test)
train_df, temp_df = train_test_split(df, test_size=0.4, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")

Number of unique StockCodes: 3665
Number of unique CustomerIDs: 4339
----------
Train set shape: (238754, 3)
Validation set shape: (79585, 3)
Test set shape: (79585, 3)


Given that StockCode and CustomerID have high cardinalities of 3,665 and 4,339 unique values respectively, one-hot encoding is highly unfeasible as it would generate over 8,000 sparse columns, drastically increasing memory consumption and training time for our 238,754 rows. Instead, we should employ more compact representation methods such as target encoding (which replaces categories with mean target values) or dense low-dimensional entity embeddings within a neural network to efficiently capture relationship patterns without the memory bottleneck.

## Part b

In [ ]:
!pip install -q category_encoders

In [ ]:
import numpy as np
from category_encoders import TargetEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

# 1. Target Transformation: Apply log1p due to heavy-tailed 'Quantity'
y_train = np.log1p(train_df['Quantity'])
y_val = np.log1p(val_df['Quantity'])
y_test = np.log1p(test_df['Quantity'])

X_train = train_df[['StockCode', 'CustomerID']].copy()
X_val = val_df[['StockCode', 'CustomerID']].copy()
X_test = test_df[['StockCode', 'CustomerID']].copy()

# Cast to string types for the category encoder
for dataset in [X_train, X_val, X_test]:
    dataset['StockCode'] = dataset['StockCode'].astype(str)
    dataset['CustomerID'] = dataset['CustomerID'].astype(str)

# 2. Fit encoder strictly on training data, then transform validation and test
encoder = TargetEncoder(cols=['StockCode', 'CustomerID'])
X_train_encoded = encoder.fit_transform(X_train, y_train)

# Using the encoding rules derived solely from the training set to prevent data leakage
X_val_encoded = encoder.transform(X_val)
X_test_encoded = encoder.transform(X_test)

# 3. Fit model (Ridge regression to demonstrate regularized fitting)
model = Ridge(alpha=1.0)
model.fit(X_train_encoded, y_train)

# 4. Predict and evaluate
val_preds = model.predict(X_val_encoded)
test_preds = model.predict(X_test_encoded)

# We measure performance against validation targets to verify generalizability before finalizing
val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))

print(f"Validation RMSE (Log Scale): {val_rmse:.4f}")
print(f"Test RMSE (Log Scale): {test_rmse:.4f}")

Validation RMSE (Log Scale): 0.6074
Test RMSE (Log Scale): 0.6087


## Part c

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 1. Map categories to unique indices (0 to N-1) with a reserved slot for unknown categories
def create_mappings(train_series):
    unique_vals = train_series.unique()
    mapping = {val: idx + 1 for idx, val in enumerate(unique_vals)}  # index 0 reserved for unknown
    return mapping, len(unique_vals) + 1

stock_mapping, num_stocks = create_mappings(train_df['StockCode'])
customer_mapping, num_customers = create_mappings(train_df['CustomerID'])

def encode_series(series, mapping):
    return series.map(mapping).fillna(0).astype(int).values

# Prepare inputs
X_train_stock = encode_series(train_df['StockCode'], stock_mapping)
X_train_cust = encode_series(train_df['CustomerID'], customer_mapping)

X_val_stock = encode_series(val_df['StockCode'], stock_mapping)
X_val_cust = encode_series(val_df['CustomerID'], customer_mapping)

X_test_stock = encode_series(test_df['StockCode'], stock_mapping)
X_test_cust = encode_series(test_df['CustomerID'], customer_mapping)

# Log-transform target 'Quantity'
y_train_arr = np.log1p(train_df['Quantity'].values).astype(np.float32)
y_val_arr = np.log1p(val_df['Quantity'].values).astype(np.float32)
y_test_arr = np.log1p(test_df['Quantity'].values).astype(np.float32)

# Create PyTorch DataLoaders
batch_size = 256
train_dataset = TensorDataset(torch.tensor(X_train_stock), torch.tensor(X_train_cust), torch.tensor(y_train_arr))
val_dataset = TensorDataset(torch.tensor(X_val_stock), torch.tensor(X_val_cust), torch.tensor(y_val_arr))
test_dataset = TensorDataset(torch.tensor(X_test_stock), torch.tensor(X_test_cust), torch.tensor(y_test_arr))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Neural Network with Embedding Layers
class EmbeddingNN(nn.Module):
    def __init__(self, num_stocks, num_customers, emb_dim_stock=50, emb_dim_cust=50):
        super(EmbeddingNN, self).__init__()
        self.stock_embed = nn.Embedding(num_stocks, emb_dim_stock)
        self.cust_embed = nn.Embedding(num_customers, emb_dim_cust)

        # Fully connected layers
        self.fc = nn.Sequential(
            nn.Linear(emb_dim_stock + emb_dim_cust, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, stock_idx, cust_idx):
        stock_feat = self.stock_embed(stock_idx)
        cust_feat = self.cust_embed(cust_idx)
        x = torch.cat([stock_feat, cust_feat], dim=1)
        return self.fc(x).squeeze(-1)

# Choose reasonable embedding dimensions
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EmbeddingNN(num_stocks, num_customers, emb_dim_stock=50, emb_dim_cust=50).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)

# 3. Training Loop with early stopping based on Validation Loss
epochs = 5
best_val_loss = float('inf')

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for stocks, custs, targets in train_loader:
        stocks, custs, targets = stocks.to(device), custs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(stocks, custs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * stocks.size(0)

    train_loss /= len(train_loader.dataset)

    # Validation step
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for stocks, custs, targets in val_loader:
            stocks, custs, targets = stocks.to(device), custs.to(device), targets.to(device)
            outputs = model(stocks, custs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * stocks.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{epochs} | Train RMSE: {np.sqrt(train_loss):.4f} | Val RMSE: {np.sqrt(val_loss):.4f}")

# 4. Evaluate on Test Set
model.eval()
test_loss = 0.0
with torch.no_grad():
    for stocks, custs, targets in test_loader:
        stocks, custs, targets = stocks.to(device), custs.to(device), targets.to(device)
        outputs = model(stocks, custs)
        loss = criterion(outputs, targets)
        test_loss += loss.item() * stocks.size(0)
test_loss /= len(test_loader.dataset)
test_rmse = np.sqrt(test_loss)

print(f"\nTest RMSE (Log Scale): {test_rmse:.4f}")

Epoch 1/5 | Train RMSE: 0.7758 | Val RMSE: 0.6233
Epoch 2/5 | Train RMSE: 0.6026 | Val RMSE: 0.5652
Epoch 3/5 | Train RMSE: 0.5705 | Val RMSE: 0.5570
Epoch 4/5 | Train RMSE: 0.5573 | Val RMSE: 0.5707
Epoch 5/5 | Train RMSE: 0.5464 | Val RMSE: 0.5419

Test RMSE (Log Scale): 0.5444


The Neural Network model with embedding layers significantly outperforms the Target Encoding model, yielding a Test RMSE of 0.5444 compared to the target encoder's Test RMSE of 0.6087 on the log-transformed target scale. This marked improvement highlights that learning dense, multi-dimensional representations of StockCode and CustomerID within an optimization loop enables the neural network to capture intricate, non-linear feature interactions that a simple, single-value-per-category target mean mapping fails to represent.

## Part d

In [ ]:
# Prepare the target-encoded features
X_train_enc_arr = X_train_encoded.values.astype(np.float32)
X_val_enc_arr = X_val_encoded.values.astype(np.float32)
X_test_enc_arr = X_test_encoded.values.astype(np.float32)

# 1. Update Datasets to include target encoded features
batch_size = 256
train_dataset = TensorDataset(
    torch.tensor(X_train_stock),
    torch.tensor(X_train_cust),
    torch.tensor(X_train_enc_arr),
    torch.tensor(y_train_arr)
)
val_dataset = TensorDataset(
    torch.tensor(X_val_stock),
    torch.tensor(X_val_cust),
    torch.tensor(X_val_enc_arr),
    torch.tensor(y_val_arr)
)
test_dataset = TensorDataset(
    torch.tensor(X_test_stock),
    torch.tensor(X_test_cust),
    torch.tensor(X_test_enc_arr),
    torch.tensor(y_test_arr)
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Hybrid Neural Network Model
class HybridEmbeddingNN(nn.Module):
    def __init__(self, num_stocks, num_customers, emb_dim_stock=50, emb_dim_cust=50):
        super(HybridEmbeddingNN, self).__init__()
        self.stock_embed = nn.Embedding(num_stocks, emb_dim_stock)
        self.cust_embed = nn.Embedding(num_customers, emb_dim_cust)

        # Concatenate: stock embedding (50) + customer embedding (50) + target encoded features (2)
        input_dim = emb_dim_stock + emb_dim_cust + 2

        self.fc = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, stock_idx, cust_idx, encoded_feats):
        stock_feat = self.stock_embed(stock_idx)
        cust_feat = self.cust_embed(cust_idx)
        x = torch.cat([stock_feat, cust_feat, encoded_feats], dim=1)
        return self.fc(x).squeeze(-1)

# Initialize model, loss, and optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
hybrid_model = HybridEmbeddingNN(num_stocks, num_customers).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(hybrid_model.parameters(), lr=0.003)

# 3. Training Loop
epochs = 5
for epoch in range(epochs):
    hybrid_model.train()
    train_loss = 0.0
    for stocks, custs, enc_feats, targets in train_loader:
        stocks, custs, enc_feats, targets = stocks.to(device), custs.to(device), enc_feats.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = hybrid_model(stocks, custs, enc_feats)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * stocks.size(0)

    train_loss /= len(train_loader.dataset)

    # Validation Step
    hybrid_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for stocks, custs, enc_feats, targets in val_loader:
            stocks, custs, enc_feats, targets = stocks.to(device), custs.to(device), enc_feats.to(device), targets.to(device)
            outputs = hybrid_model(stocks, custs, enc_feats)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * stocks.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{epochs} | Train RMSE: {np.sqrt(train_loss):.4f} | Val RMSE: {np.sqrt(val_loss):.4f}")

# 4. Evaluation on Test Set
hybrid_model.eval()
test_loss = 0.0
with torch.no_grad():
    for stocks, custs, enc_feats, targets in test_loader:
        stocks, custs, enc_feats, targets = stocks.to(device), custs.to(device), enc_feats.to(device), targets.to(device)
        outputs = hybrid_model(stocks, custs, enc_feats)
        loss = criterion(outputs, targets)
        test_loss += loss.item() * stocks.size(0)
test_loss /= len(test_loader.dataset)
test_rmse = np.sqrt(test_loss)

print(f"\nCombined Model Test RMSE (Log Scale): {test_rmse:.4f}")

Epoch 1/5 | Train RMSE: 0.7012 | Val RMSE: 0.5991
Epoch 2/5 | Train RMSE: 0.6127 | Val RMSE: 0.5740
Epoch 3/5 | Train RMSE: 0.5785 | Val RMSE: 0.5609
Epoch 4/5 | Train RMSE: 0.5565 | Val RMSE: 0.5448
Epoch 5/5 | Train RMSE: 0.5408 | Val RMSE: 0.5419

Combined Model Test RMSE (Log Scale): 0.5451


The hybrid model—which combines target encoded features directly with learned categorical entity embeddings—achieved a Test RMSE of 0.5451 on the log scale. This performance is virtually identical to (and slightly trails) the pure Embedding Neural Network (0.5444), while still substantially outperforming the simple Target Encoding regression baseline (0.6087). This indicates that adding the single-dimensional target encoded mean statistics does not provide any additive predictive value or residual information beyond what the expressive, 50-dimensional categorical embedding layers already learn and represent end-to-end within the network.

## Part e

#### 1. What is the Embedding Layer doing?
An embedding layer acts as a parameterized lookup table that maps high-cardinality discrete IDs (like individual `StockCode`s or `CustomerID`s) into a continuous, dense, and low-dimensional vector space (50 dimensions in our case). Unlike discrete encodings (one-hot or target encoding) which treat each category as completely isolated or compress it down to a single number, embedding layers represent each category with a distributed representation. During training, the coordinates of these 50-dimensional vectors are adjusted via backpropagation to minimize the target prediction error.

#### 2. Why is it useful in this context?
* **Capturing Hidden Semantic Similarities:** The embedding space automatically places similar entities close to each other. For example, two different `StockCodes` representing complementary items (e.g., a teacup and a saucer) or two `CustomerID`s with similar wholesale buying profiles will end up having very similar vector directions.
* **Learning Complex Multi-dimensional Interactions:** Since the neural network receives the concatenated 50D embeddings of both the customer and the stock, it can learn non-linear combinations through its dense hidden layers (e.g., *'Is this specific type of customer likely to buy a very large quantity of this specific type of item?'*). A single scalar target encoding cannot capture these joint relational patterns.

#### 3. Test Set Evaluation Summary
* **Target Encoding Baseline (Ridge Regression):** Test RMSE of **0.6087**
* **Pure Embedding Neural Network:** Test RMSE of **0.5444**
* **Combined/Hybrid Neural Network:** Test RMSE of **0.5451**

**Conclusion:** Moving from simple target encoding to learned embeddings yielded a major performance improvement (~10.5% reduction in prediction error). Adding target encodings to the embeddings (the hybrid model) did not improve performance further because the embedding layer's 50-dimensional space had already captured all the variance and statistical signals represented by the target mean.